# 04 — Preprocessing Pipeline
Build a ColumnTransformer, demonstrate data leakage prevention, compare scalers.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from src.preprocess import build_preprocessor
from src.config import NUMERICAL_FEATURES, CATEGORICAL_FEATURES, RANDOM_STATE

## 1. Load Split Data

In [ ]:
X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
y_train = pd.read_csv('data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('data/processed/y_test.csv').values.ravel()
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## 2. Build and Inspect Preprocessor

In [ ]:
preprocessor = build_preprocessor()
print(preprocessor)

## 3. Data Leakage Demonstration
Leakage occurs when test data influences training. Here we show the difference
between fitting a scaler on ALL data vs train-only.

In [ ]:
from sklearn.preprocessing import StandardScaler

# WRONG: fit on ALL data (leaks test distribution into training)
scaler_leak = StandardScaler()
all_income = pd.concat([X_train['Income'], X_test['Income']])
scaler_leak.fit(all_income.values.reshape(-1, 1))
leaked_mean = scaler_leak.mean_[0]

# CORRECT: fit on train only
scaler_correct = StandardScaler()
scaler_correct.fit(X_train['Income'].values.reshape(-1, 1))
correct_mean = scaler_correct.mean_[0]

print(f"Leaked scaler mean (fit on all):    {leaked_mean:,.2f}")
print(f"Correct scaler mean (fit on train): {correct_mean:,.2f}")
print(f"Difference: {abs(leaked_mean - correct_mean):,.2f}")
print("\n→ In sklearn Pipeline, the preprocessor is fit ONLY on train data inside .fit()")

## 4. Full Pipeline with LogisticRegression

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

pipeline.fit(X_train, y_train)
print(f"Pipeline score on test: {pipeline.score(X_test, y_test):.4f}")
print("\n✓ Preprocessor fitted ONLY on training data inside the pipeline")

## 5. Key Takeaway
The pipeline ensures:
1. Preprocessor is fit on train data ONLY
2. Same transformations are applied to both train and test
3. The entire pipeline can be pickled as one object
4. No data leakage is possible